# BCB statement-only input-domain review
Public-contract adjudications: BCB65 permits arbitrary row values because the statement specifies row shape/grouping, not numeric element types; BCB87 permits zero individual weights when weights are finite, nonnegative, and not all zero; BCB95 permits empty strings inside a nonempty list of strings. These decisions use no execution behavior.
This offline review reads only public task specifications and generated function-mode kwargs. It never loads candidate code, reference solutions, labels, secret inputs, model prompts, or execution outcomes. It records every input as `valid`, `invalid`, or `unresolved` with a reason; unresolved semantics block the reviewed-input freeze rather than being silently dropped. It requires all 52 trigger records before writing the immutable review and reviewed-input artifacts.


In [1]:
from pathlib import Path
import hashlib, json, math, re, sys
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); sys.path.insert(0,str(REPO)); import os; os.chdir(REPO)
DATA=Path('data/bcb_replication26_eval.json')
if not DATA.exists(): DATA=Path('data/bcb.json')
PREFIX='azure-terra-pbt-bcb26-s300-v1'; SOURCE_RUN=PREFIX+'-triggers'; SOURCE_DIR=Path('runs')/SOURCE_RUN
REVIEW_PATH=SOURCE_DIR/'domain-review-v4.json'; REVIEWED_RUN=PREFIX+'-reviewed-inputs'; REVIEWED_DIR=Path('runs')/REVIEWED_RUN
def sha(raw): return hashlib.sha256(raw if isinstance(raw,bytes) else raw.encode()).hexdigest()
def file_sha(p): return sha(Path(p).read_bytes())
def immutable(path, doc):
    raw=(json.dumps(doc,sort_keys=True,indent=2,ensure_ascii=False)+'\n').encode()
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists(): assert path.read_bytes()==raw, f'immutable artifact differs: {path}'
    else: path.write_bytes(raw)
def immutable_text(path, raw):
    raw=raw.encode() if isinstance(raw,str) else raw
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists(): assert path.read_bytes()==raw, f'immutable artifact differs: {path}'
    else: path.write_bytes(raw)
doc=json.loads(DATA.read_text(encoding='utf-8')); assert len(doc['tasks'])==26
specs={t['task_id']:t['specification'] for t in doc['tasks']}
print({'tasks':len(specs),'source_records':str(SOURCE_DIR/'records.jsonl'),'model_calls':0,'candidate_execution':0})


{'tasks': 26, 'source_records': 'runs\\azure-terra-pbt-bcb26-s300-v1-triggers\\records.jsonl', 'model_calls': 0, 'candidate_execution': 0}


In [2]:
def json_value(value):
    if value is None or isinstance(value,(str,bool,int,float)): return not isinstance(value,float) or math.isfinite(value)
    if isinstance(value,list): return all(json_value(x) for x in value)
    if isinstance(value,dict): return all(isinstance(k,str) and json_value(v) for k,v in value.items())
    return False
def _num(x): return isinstance(x,(int,float)) and not isinstance(x,bool) and math.isfinite(x) if isinstance(x,float) else isinstance(x,(int,float)) and not isinstance(x,bool)
def _strlist(x): return isinstance(x,list) and all(isinstance(y,str) for y in x)
def _numlist(x): return isinstance(x,list) and all(_num(y) for y in x)
def _pairs(x): return isinstance(x,list) and all(isinstance(y,list) and len(y)==2 and isinstance(y[0],str) and _num(y[1]) for y in x)
def _triples(x): return isinstance(x,list) and all(isinstance(y,list) and len(y)==3 for y in x)
def statement_review(task_id,spec,value):
    if not isinstance(value,dict): return 'invalid','function-mode input must be a JSON object of keyword arguments'
    if not json_value(value): return 'invalid','input contains a non-JSON value'
    # Derive the public callable signature without importing or executing task code.
    import ast
    try:
        tree=ast.parse(spec); fn=next(n for n in tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name=='task_func')
        positional=[a.arg for a in fn.args.posonlyargs+fn.args.args]; defaults=[None]*(len(positional)-len(fn.args.defaults))+[d for d in fn.args.defaults]; names=positional+[a.arg for a in fn.args.kwonlyargs]
        required={n for n,d in zip(positional,defaults) if d is None} | {a.arg for a,d in zip(fn.args.kwonlyargs,fn.args.kw_defaults) if d is None}
    except Exception as e: return 'unresolved',f'public signature could not be parsed: {type(e).__name__}'
    unknown=set(value)-set(names)
    if unknown: return 'invalid',f'unknown function keyword(s): {sorted(unknown)}'
    if required-set(value): return 'invalid',f'missing required keyword(s): {sorted(required-set(value))}'
    tid=task_id.rsplit('/',1)[-1]
    # Task-specific statements are explicit public input contracts; ambiguous behavior remains unresolved.
    if tid=='27' and isinstance(value.get('data'),dict) and 'timestamp' in value['data']: return 'invalid','data must not contain timestamp'
    if tid=='27' and (not isinstance(value.get('data'),dict) or not isinstance(value.get('DATE_FORMAT','%Y-%m-%d %H:%M:%S'),str)): return 'invalid','data must be a dict and DATE_FORMAT a string'
    if tid=='141' and ('rows' in value) and (not isinstance(value['rows'],int) or isinstance(value['rows'],bool) or value['rows']<=0): return 'invalid','rows must be a positive integer'
    if tid=='141' and (not _strlist(value.get('columns')) or not isinstance(value.get('seed'),int) or isinstance(value.get('seed'),bool)): return 'invalid','columns must be strings and seed an integer'
    if tid=='84':
        if not _strlist(value.get('products')): return 'invalid','products must be a list of strings'
        if not isinstance(value.get('n_samples'),int) or isinstance(value.get('n_samples'),bool): return 'invalid','n_samples must be an integer'
        for k in ('sales_lower','sales_upper'):
            if not _num(value.get(k)): return 'invalid',f'{k} must be numeric'
        if value['n_samples']<=0: return 'invalid','n_samples must be positive'
        if value['sales_lower']>value['sales_upper']: return 'invalid','sales_lower exceeds sales_upper'
    if tid=='95':
        for k in ('categories','months'):
            if value.get(k) is not None and (not _strlist(value[k]) or not value[k]): return 'invalid',f'{k} must be null or a nonempty list of strings'
    if tid in {'121','149','153','63','86'}:
        key={'121':'my_list','149':'elements','153':'data','63':'car_dict','86':'students'}[tid]; x=value.get(key)
        if tid=='63' and (not isinstance(x,dict) or not all(isinstance(k,str) and isinstance(v,str) for k,v in x.items())): return 'invalid','car_dict must map strings to strings'
        if tid!='63' and not (_strlist(x) if tid in {'149','153','86'} else isinstance(x,list)): return 'invalid',f'{key} has the wrong public type'
    if tid=='149' and not isinstance(value.get('include_index'),bool): return 'invalid','include_index must be boolean'
    if tid in {'121','86'} and not (isinstance(value.get('seed'),int) and not isinstance(value.get('seed'),bool)): return 'invalid','seed must be an integer'
    if tid in {'52','54','55'} and not isinstance(value.get('text'),str): return 'invalid','text must be a string'
    if tid=='50' and (not isinstance(value.get('timestamp'),int) or isinstance(value.get('timestamp'),bool)): return 'invalid','timestamp must be an integer'
    if tid=='147':
        import ipaddress
        try: ipaddress.ip_network(value.get('ip_range'),strict=False)
        except Exception: return 'invalid','ip_range must be valid CIDR notation'
    if tid in {'4'} and (not isinstance(value.get('d'),dict) or not all(isinstance(k,str) and _numlist(v) for k,v in value['d'].items())): return 'invalid','d must map strings to integer lists'
    if tid in {'9','33'} and not _pairs(value.get('list_of_pairs')): return 'invalid','list_of_pairs must contain string/numeric pairs'
    if tid in {'64','65','66'} and not _triples(value.get('data')): return 'invalid','data must contain rows of length three'
    if tid=='61':
        if not isinstance(value.get('result'),list): return 'invalid','result must be a list'
        if any(not isinstance(x,dict) or ('from_user' in x and (not _num(x['from_user']) or x['from_user']<0)) for x in value['result']): return 'invalid','from_user values must be nonnegative numbers'
    if tid=='147':
        if not isinstance(value.get('ip_range'),str) or not isinstance(value.get('port'),int) or not 0<=value['port']<=65535: return 'invalid','ip_range/port violate the public socket input types'
    if tid=='87':
        if not (_strlist(value.get('products')) and _numlist(value.get('ratings')) and _numlist(value.get('weights'))): return 'invalid','products/ratings/weights have wrong types'
        if len(value['ratings'])!=len(value['weights']) or any(x<0 or not math.isfinite(x) for x in value['weights']) or sum(value['weights'])<=0: return 'invalid','weights must be finite, nonnegative, and match ratings'
    if tid=='97' and (not isinstance(value.get('numbers'),list) or any(not isinstance(x,int) or isinstance(x,bool) or x<=0 for x in value['numbers'])): return 'invalid','numbers must be a nonempty list of positive integers'
    if tid=='4' and any(any(not isinstance(y,int) or isinstance(y,bool) for y in v) for v in value.get('d',{}).values()): return 'invalid','d values must be integer lists'
    if tid in {'9','33'} and any(not isinstance(y[1],int) or isinstance(y[1],bool) for y in value.get('list_of_pairs',[])): return 'invalid','pair values must be integers'
    if tid=='150' and any(not (isinstance(v,list) and len(v)==2 and _num(v[0]) and _num(v[1])) for v in value.get('product_dict',{}).values()): return 'invalid','product values must be [quantity, price] numeric pairs'
    if tid=='151':
        d=value.get('data_dict',{}); keys=value.get('data_keys',[])
        if not isinstance(d,dict) or not _strlist(keys) or any(not _numlist(v) for v in d.values()): return 'invalid','data_dict/data_keys have wrong public types'
        found=[k for k in keys if k in d]
        if not found: return 'invalid','data_keys must contain at least one data_dict key'
        if len({len(d[k]) for k in found})!=1: return 'invalid','selected data lists must have equal lengths'
    if tid in {'25','150','151'} and not isinstance(value.get('data_dict' if tid in {'25','151'} else 'product_dict'),dict): return 'invalid','dictionary input has wrong type'
    return 'valid','matches the public signature and stated input-domain contract'
def review_record(row):
    task_id=str(row.get('task_id')); spec=specs.get(task_id)
    if spec is None: return {'status':'invalid','reason':'task id absent from frozen public dataset'}
    if row.get('failed'): return {'status':'unresolved','reason':'trigger record failed; no input decision fabricated'}
    decisions=[]
    for index,value in enumerate(row.get('inputs',[])):
        status,reason=statement_review(task_id,spec,value); decisions.append({'index':index,'input':value,'status':status,'reason':reason})
    return {'decisions':decisions,'valid_indices':[d['index'] for d in decisions if d['status']=='valid']}


In [3]:
source_path=SOURCE_DIR/'records.jsonl'; rows=[]
if source_path.exists():
    for line_no,rawline in enumerate(source_path.read_bytes().splitlines(keepends=True)):
        raw=rawline.rstrip(b'\r\n')
        if raw.strip():
            row=json.loads(raw); rows.append({'source_record_index':line_no,'source_record_sha256':sha(raw),
                'task_id':row.get('task_id'),'candidate_id':row.get('candidate_id'),'failed':row.get('failed'),
                'n_requested':row.get('n_requested'),'n_parsed':row.get('n_parsed'),'dropped':row.get('dropped'),
                'call_metadata':[{'rep':c.get('rep'),'seed':c.get('seed'),'usage':c.get('usage')} for c in row.get('calls',[])],
                'calls_sha256':sha(json.dumps(row.get('calls',[]),sort_keys=True,separators=(',',':'))),
                'inputs':row.get('inputs',[])})
coverage={'records':len(rows),'expected':52,'duplicate_candidate_ids':len(rows)!=len({r['candidate_id'] for r in rows})}
assert len(rows)<=52 and not coverage['duplicate_candidate_ids']
reviewed=[]
for row in rows:
    decision=review_record(row); reviewed.append({k:row[k] for k in ('source_record_index','source_record_sha256','task_id','candidate_id','call_metadata','calls_sha256')}|decision)
counts={s:sum(d.get('status')==s for d in (x for r in reviewed for x in r.get('decisions',[]))) for s in ('valid','invalid','unresolved')}
review={'schema_version':1,'status':'ready' if len(rows)==52 and counts['unresolved']==0 and all(r['valid_indices'] for r in reviewed) else 'blocked',
 'dataset_sha256':file_sha(DATA),'source_records_sha256':file_sha(source_path) if source_path.exists() else None,
 'candidate_count':52,'coverage':coverage,'counts':counts,'records':reviewed,
 'acceptance':'statement-only public specification; no candidate/reference/label/outcome access'}
immutable(REVIEW_PATH,review)
print({'review_status':review['status'],'coverage':coverage,'counts':counts})


{'review_status': 'ready', 'coverage': {'records': 52, 'expected': 52, 'duplicate_candidate_ids': False}, 'counts': {'valid': 517, 'invalid': 3, 'unresolved': 0}}


In [4]:
if review['status']=='ready':
    accepted=[]
    for r in reviewed:
        for d in r['decisions']:
            if d['status']=='valid': accepted.append({'candidate_id':r['candidate_id'],'task_id':r['task_id'],'index':d['index'],'input':d['input']})
    assert len({r['candidate_id'] for r in reviewed})==52 and all(r['valid_indices'] for r in reviewed)
    reviewed_doc={'schema_version':1,'run_name':REVIEWED_RUN,'dataset_sha256':file_sha(DATA),
      'source_records_sha256':file_sha(source_path),'domain_review_sha256':file_sha(REVIEW_PATH),
      'candidate_input_sha256':{r['candidate_id']:sha(json.dumps([x['input'] for x in accepted if x['candidate_id']==r['candidate_id']],sort_keys=True,separators=(',',':'))) for r in reviewed},
      'records':[{'candidate_id':r['candidate_id'],'task_id':r['task_id'],'inputs':[x['input'] for x in accepted if x['candidate_id']==r['candidate_id']],
        'source_record_index':r['source_record_index'],'source_record_sha256':r['source_record_sha256']} for r in reviewed]}
    immutable(REVIEWED_DIR/'reviewed-inputs.json',reviewed_doc)
    immutable(REVIEWED_DIR/'config.json',{'run_name':REVIEWED_RUN,'protocol':'statement_domain_review','split':'test','source_run':SOURCE_RUN,'dataset_sha256':file_sha(DATA),'domain_review_sha256':file_sha(REVIEW_PATH),'new_model_calls':0,'candidate_execution':0})
    record_lines=[]
    for r in reviewed:
        selected=[d['input'] for d in r['decisions'] if d['status']=='valid']
        rec={'run_name':REVIEWED_RUN,'protocol':'statement_domain_review','split':'test','task_id':r['task_id'],'candidate_id':r['candidate_id'],'failed':False,'reason':'','inputs':selected,'n_requested':len(selected),'n_parsed':len(selected),'dropped':0,'calls':[],'source_run':SOURCE_RUN,'source_record_index':r['source_record_index'],'source_record_sha256':r['source_record_sha256'],'provenance':{'source_dataset_sha256':file_sha(DATA),'source_records_sha256':file_sha(source_path),'review_path':str(REVIEW_PATH),'review_sha256':file_sha(REVIEW_PATH),'source_input_indices':[d['index'] for d in r['decisions'] if d['status']=='valid'],'transformation':'statement-only selection of existing generated inputs; no new generation','new_model_calls':0}}
        record_lines.append(json.dumps(rec,sort_keys=True,separators=(',',':')))
    immutable_text(REVIEWED_DIR/'records.jsonl','\n'.join(record_lines)+'\n')
    review['input_records_sha256']=file_sha(REVIEWED_DIR/'records.jsonl')
    review['candidate_input_sha256']={r['candidate_id']:sha(json.dumps([d['input'] for d in r['decisions'] if d['status']=='valid'],sort_keys=True,separators=(',',':'))) for r in reviewed}
    review['all_candidates_resolved']=bool(len(rows)==52 and counts['unresolved']==0 and all(r['valid_indices'] for r in reviewed))
    immutable(REVIEWED_DIR/'statement-review-v1.json',review)
    print({'reviewed_artifact':'frozen','candidates':52,'accepted_inputs':len(accepted),'records_path':str(REVIEWED_DIR/'records.jsonl')})
else:
    review['all_candidates_resolved']=False
    if (REVIEWED_DIR/'records.jsonl').exists(): review['input_records_sha256']=file_sha(REVIEWED_DIR/'records.jsonl')
    review['candidate_input_sha256']={r['candidate_id']:sha(json.dumps([d['input'] for d in r['decisions'] if d['status']=='valid'],sort_keys=True,separators=(',',':'))) for r in reviewed}
    immutable(REVIEWED_DIR/'statement-review-v1.json',review)
    print({'reviewed_artifact':'blocked','reason':'unresolved statement semantics require adjudication before freeze'})


{'reviewed_artifact': 'frozen', 'candidates': 52, 'accepted_inputs': 517, 'records_path': 'runs\\azure-terra-pbt-bcb26-s300-v1-reviewed-inputs\\records.jsonl'}


Review policy: the artifact freezes only after all 52 trigger records are present and every generated input has a decision. It never substitutes missing calls with clean inputs, never accepts explicit invalid cases (exceptions are candidate crashes, not catches), and never uses candidate/reference code, labels, prompts, or outcomes for acceptance.
